In [2]:
import pandas as pd

# Step 1: Read the second Excel file and build the feature mapping
file2 = pd.ExcelFile('240715-NTA-Paper2-KARL-RFE-TARGETS-ANNOTATED-LIB-SCORE-DESCRIPTIONS.xlsx')
df2 = file2.parse(file2.sheet_names[0])  # Only one sheet in this file

feature_mapping = {}
num_trios = 6  # There are six trios of columns

for i in range(num_trios):
    col_idx = i * 3
    feature_col = df2.columns[col_idx]
    chem_col = df2.columns[col_idx + 1]
    desc_col = df2.columns[col_idx + 2]

    features = df2[feature_col].astype(str).str.strip()
    chems = df2[chem_col].astype(str).str.strip()
    descs = df2[desc_col].astype(str).str.strip()

    for feature, chem, desc in zip(features, chems, descs):
        if feature and feature != 'nan' and feature != 'NA':
            feature_mapping[feature] = (chem, desc)

# Step 2: Process each sheet in the first Excel file
file1 = pd.ExcelFile('240907-NTA-Paper2-RFE-AnnotatedFeatures-ONLY-SUMMARY.xlsx')
output_writer = pd.ExcelWriter('output.xlsx', engine='xlsxwriter')

for sheet_name in file1.sheet_names:
    df1 = file1.parse(sheet_name)
    df1 = df1.fillna('NA')  # Replace NaNs with 'NA'

    # Prepare data for the new DataFrame
    data = {}
    new_columns = []

    for col in df1.columns:
        feature_col = col
        chem_col = f'{col}_ChemName'
        desc_col = f'{col}_Description'

        new_columns.extend([feature_col, chem_col, desc_col])

        features = df1[col].astype(str).str.strip().tolist()
        chem_names = []
        descriptions = []

        for feature in features:
            if feature == 'NA' or feature == '' or feature == 'nan':
                chem_names.append('NA')
                descriptions.append('NA')
            else:
                match = feature_mapping.get(feature)
                if match:
                    chem_names.append(match[0])
                    descriptions.append(match[1])
                else:
                    chem_names.append('NA')
                    descriptions.append('NA')

        data[feature_col] = features
        data[chem_col] = chem_names
        data[desc_col] = descriptions

    # Step 3: Create a new DataFrame with the results
    new_df = pd.DataFrame(data, columns=new_columns)
    new_df.to_excel(output_writer, sheet_name=sheet_name, index=False)

# Step 4: Save the results to a new Excel file
output_writer.save()